# ERP pattern example browser

Full-resolution real-data ERP images from the project's manual Label Studio labels. The notebook selects at most 20 examples per pattern class and plots every ERP image as its own figure with its own colour bar.


## 1. Load annotations and dependencies


In [ ]:
using CairoMakie
using DataFrames

function find_repo_root(start_dir::AbstractString = pwd())
    candidates = unique(normpath.([
        start_dir,
        joinpath(start_dir, ".."),
        joinpath(start_dir, "..", ".."),
        joinpath(start_dir, "..", "..", ".."),
    ]))
    for candidate in candidates
        if isdir(joinpath(candidate, "notebooks")) && isdir(joinpath(candidate, "scripts"))
            return candidate
        end
    end
    error("Could not locate repository root from start_dir=$(start_dir).")
end

const REPO_ROOT = find_repo_root()
const PATTERN_SCRIPT = joinpath(REPO_ROOT, "notebooks", "week_23", "erp_pattern_examples.jl")

include(PATTERN_SCRIPT)
using .Week23ERPPatternExamples

CairoMakie.activate!(type = "png")

annotations = load_labelled_annotations()
available_examples_by_class(annotations)


## 2. Select labelled examples per pattern class


In [ ]:
selected_labels = select_labelled_examples(
    annotations;
    examples_per_class = EXAMPLES_PER_CLASS,
    seed = RNG_SEED,
)

selected_labels[:, [
    :erp_class,
    :dataset_key,
    :channel_name,
    :sort_variable,
    :figure_row,
    :figure_col,
    :annotation_id,
]]


## 3. Reconstruct full-resolution ERP images


In [ ]:
samples = reconstruct_full_resolution_examples(selected_labels)
selected_table = selected_examples_table(samples)

selected_table[:, [
    :pattern_class,
    :dataset_key,
    :channel,
    :sort_variable,
    :n_trials,
    :n_timepoints,
    :time_start_s,
    :time_end_s,
    :baseline_correct,
]]


## 4. Plot individual ERP images by class

Each cell below displays separate figures. Each ERP image has its own colour range and colour bar.


### Sigmoid


In [ ]:
display_class_examples(samples, "sigmoid")


### Tilted Bar


In [ ]:
display_class_examples(samples, "tilted_bar")


### One-sided Fan


In [ ]:
display_class_examples(samples, "one_sided_fan")


### Two-sided Fan


In [ ]:
display_class_examples(samples, "two_sided_fan")


### Diverging Bar


In [ ]:
display_class_examples(samples, "diverging_bar")


### Hourglass


In [ ]:
display_class_examples(samples, "hourglass")


## 5. Print selected examples table


In [ ]:
selected_table


## 6. Export thesis SVGs


In [ ]:
thesis_labels = select_examples_by_specs(annotations, THESIS_EXPORT_SPECS)
thesis_samples = reconstruct_full_resolution_examples(thesis_labels)

exported_svg_paths = export_individual_erp_svgs(
    thesis_samples;
    output_dir = THESIS_EXPORT_DIR,
    formats = ("svg",),
)

(
    exported_svg_paths = exported_svg_paths,
    exported_examples = selected_examples_table(thesis_samples),
)
